<a href="https://colab.research.google.com/github/niranjanniru-max/Titanic-Survival-ipynb/blob/main/Titanic_Survival_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
import pandas as pd
import os

# The name of the specific file within the dataset you want to load
# For 'bhanupratapbiswas/titanic-survival-datasets', 'train.csv' is a common choice.
file_name = "titanic.csv"

# Download the dataset. This caches the dataset locally and returns the path to its directory.
dataset_dir = kagglehub.dataset_download(
  "bhanupratapbiswas/titanic-survival-datasets"
)

# List contents of the downloaded directory to find the correct file path
print(f"Contents of dataset directory: {os.listdir(dataset_dir)}")

# Construct the full path to the specific file
file_path = os.path.join(dataset_dir, file_name)

# Load the dataset into a pandas DataFrame
data = pd.read_csv(file_path)

Using Colab cache for faster access to the 'titanic-survival-datasets' dataset.
Contents of dataset directory: ['titanic.csv']


In [ ]:
data.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [ ]:
data.drop(['PassengerId','Name'], axis=1,inplace=True)

In [ ]:
print(data.shape)
print(data.columns.isnull().sum())
print(data.dtypes)
print(data.describe())
print(data['Survived'].value_counts())
print(data['Sex'].unique())
print(data['Embarked'].unique())

(418, 10)
0
Survived      int64
Pclass        int64
Sex          object
Age         float64
SibSp         int64
Parch         int64
Ticket       object
Fare        float64
Cabin        object
Embarked     object
dtype: object
         Survived      Pclass         Age       SibSp       Parch        Fare
count  418.000000  418.000000  332.000000  418.000000  418.000000  417.000000
mean     0.363636    2.265550   30.272590    0.447368    0.392344   35.627188
std      0.481622    0.841838   14.181209    0.896760    0.981429   55.907576
min      0.000000    1.000000    0.170000    0.000000    0.000000    0.000000
25%      0.000000    1.000000   21.000000    0.000000    0.000000    7.895800
50%      0.000000    3.000000   27.000000    0.000000    0.000000   14.454200
75%      1.000000    3.000000   39.000000    1.000000    0.000000   31.500000
max      1.000000    3.000000   76.000000    8.000000    9.000000  512.329200
Survived
0    266
1    152
Name: count, dtype: int64
['male' 'female']
[

In [ ]:
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,accuracy_score
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,IterativeImputer

In [ ]:
# Removed global LabelEncoder for 'Sex' to prevent potential leakage and integrate into pipeline.

In [ ]:
X=data.drop(['Survived'],axis=1) # Keep 'Sex' as an object column for ColumnTransformer
y=data['Survived']

# Redefine num and cat based on the updated X
num=X.select_dtypes(exclude='object').columns
cat=X.select_dtypes(include='object').columns

print(X.columns.isnull().sum())

0


In [ ]:
numeric_transformer = Pipeline([
    ('imputer', IterativeImputer(random_state=42)),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num),
    ('cat', categorical_transformer, cat) # 'Sex' and 'Embarked' will now be handled here
])

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

pipe=Pipeline(
    [
        ('preprocessor',preprocessor),
        ('classifier',LogisticRegression(solver='liblinear')) # Added solver for LogisticRegression
    ]
)

In [ ]:
data.columns.isnull().sum()

np.int64(0)

In [ ]:
pipe.fit(X_train,y_train)
y_pred=pipe.predict(X_test)
print(classification_report(y_test,y_pred))
print(accuracy_score(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      1.00      1.00        34

    accuracy                           1.00        84
   macro avg       1.00      1.00      1.00        84
weighted avg       1.00      1.00      1.00        84

1.0


**My version is production‑ready while theirs is demo‑style. They focused on simple preprocessing, one model, and plots for explanation. I built a proper ML pipeline, compared multiple models, tuned hyperparameters, and reported structured metrics. Mine is closer to real‑world practice, theirs is more classroom‑oriented.**

In [ ]:
print(X.shape)
print(y.value_counts())

print(pipe.score(X_train,y_train))
print(pipe.score(X_test,y_test))

(418, 9)
Survived
0    266
1    152
Name: count, dtype: int64
1.0
1.0


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    pipe,
    X,
    y,
    cv=5
)

print(scores)
print(scores.mean())

[1. 1. 1. 1. 1.]
1.0


In [ ]:
print(data.columns.tolist())

['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
